In [17]:
import pandas as pd
import numpy as np
import os

#!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import warnings
warnings.filterwarnings('ignore')

from sklearn.utils.class_weight import compute_class_weight

In [12]:
print("="*70)
print("문서 타입 분류 - 완전한 베이스라인 모델")
print("="*70)

# 1단계: 데이터 로드
print("\\n[1단계] 데이터 로드 중...")
train_df = pd.read_csv('train.csv')
sample_submission = pd.read_csv('sample_submission.csv')
meta_df = pd.read_csv('meta.csv')

print(f"✓ 학습 데이터: {len(train_df)}개")
print(f"✓ 테스트 데이터: {len(sample_submission)}개")
print(f"✓ 클래스 수: {len(meta_df)}개")

# 2단계: 이미지 전처리 함수
def load_and_preprocess(img_path, target_size=224):
    try:
        img = load_img(img_path, target_size=(target_size, target_size))
        img_array = img_to_array(img)
        img_array = img_array / 255.0
        return img_array
    except:
        return np.zeros((target_size, target_size, 3))

# 3단계: 학습 데이터 로드
print("\\n[2단계] 학습 이미지 로드 중...")
train_df['file_path'] = train_df['ID'].apply(lambda x: os.path.join('train', x))

X_train = []
for idx, (_, row) in enumerate(train_df.iterrows()):
    if idx % 200 == 0:
        print(f"  진행: {idx}/{len(train_df)}")
    X_train.append(load_and_preprocess(row['file_path']))

X_train = np.array(X_train)
y_train = train_df['target'].values
print(f"✓ X_train shape: {X_train.shape}")

# 4단계: 모델 구축
print("\\n[3단계] 모델 구축 중...")

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(17, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(f"✓ 모델 생성 완료")

# 5단계: 모델 학습
print("\\n[4단계] 모델 학습 시작 (5 에포크)...")
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.2, verbose=1)
print("✓ 학습 완료")

# 6단계: 테스트 데이터 예측
print("\\n[5단계] 테스트 이미지 예측 중...")

test_files = sorted([f for f in os.listdir('test') if f.endswith(('.jpg', '.png', '.jpeg'))])
test_image_paths = [os.path.join('test', f) for f in test_files]
print(f"✓ 테스트 이미지 수: {len(test_image_paths)}")

batch_size = 64
predictions_list = []

for i in range(0, len(test_image_paths), batch_size):
    if i % 320 == 0:
        print(f"  진행: {min(i+batch_size, len(test_image_paths))}/{len(test_image_paths)}")
    batch_paths = test_image_paths[i:i+batch_size]
    batch_images = np.array([load_and_preprocess(path) for path in batch_paths])
    batch_preds = model.predict(batch_images, verbose=0)
    predictions_list.append(batch_preds)

all_predictions = np.vstack(predictions_list)
predicted_classes = np.argmax(all_predictions, axis=1)
print(f"✓ 예측 완료")

# 7단계: 제출 파일 생성
print("\\n[6단계] 제출 파일 생성 중...")
sample_submission['target'] = predicted_classes
output_path = 'submission_for_test1.csv'
sample_submission.to_csv(output_path, index=False)
print(f"✓ 저장 완료: {output_path}")

문서 타입 분류 - 완전한 베이스라인 모델
\n[1단계] 데이터 로드 중...
✓ 학습 데이터: 1570개
✓ 테스트 데이터: 3140개
✓ 클래스 수: 17개
\n[2단계] 학습 이미지 로드 중...
  진행: 0/1570
  진행: 200/1570
  진행: 400/1570
  진행: 600/1570
  진행: 800/1570
  진행: 1000/1570
  진행: 1200/1570
  진행: 1400/1570
✓ X_train shape: (1570, 224, 224, 3)
\n[3단계] 모델 구축 중...


E0000 00:00:1761896259.945694    8089 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1761896260.014798    8089 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✓ 모델 생성 완료
\n[4단계] 모델 학습 시작 (5 에포크)...
Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.0653 - loss: 2.8687 - val_accuracy: 0.0446 - val_loss: 2.8476
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 39s 979ms/step - accuracy: 0.0653 - loss: 2.8359 - val_accuracy: 0.0828 - val_loss: 2.8243
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 39s 980ms/step - accuracy: 0.0653 - loss: 2.8268 - val_accuracy: 0.0541 - val_loss: 2.8262
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 39s 980ms/step - accuracy: 0.0629 - loss: 2.8265 - val_accuracy: 0.0637 - val_loss: 2.8266
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 39s 985ms/step - accuracy: 0.0669 - loss: 2.8206 - val_accuracy: 0.0828 - val_loss: 2.8242
✓ 학습 완료
\n[5단계] 테스트 이미지 예측 중...
✓ 테스트 이미지 수: 3140
  진행: 64/3140
  진행: 384/3140
  진행: 704/3140
  진행: 1024/3140
  진행: 1344/3140
  진행: 1664/3140
  진행: 1984/3140
  진행: 2304/3140
  진행: 2624/3140
  진행: 2944/3140
✓ 예측 완료
\n[6단계] 제출 파일 생성 중...
✓ 저장 완료: submission_for_test1.csv


In [14]:
# 8단계: 최종 검증
print("\\n=== 최종 검증 ===")
print(f"✓ Total samples: {len(sample_submission)}")
print(f"✓ Columns: {list(sample_submission.columns)}")
print(f"✓ Target range: {sample_submission['target'].min()} - {sample_submission['target'].max()}")
print(f"✓ No missing values: {sample_submission.isnull().sum().sum() == 0}")
print("\\n클래스 분포:")
print(sample_submission['target'].value_counts().sort_index())
print("\\n첫 10개 행:")
print(sample_submission.head(10))
print("\\n" + "="*70)
print("✓ 제출 준비 완료!")
print("="*70)

\n=== 최종 검증 ===
✓ Total samples: 3140
✓ Columns: ['ID', 'target']
✓ Target range: 15 - 15
✓ No missing values: True
\n클래스 분포:
target
15    3140
Name: count, dtype: int64
\n첫 10개 행:
                     ID  target
0  0008fdb22ddce0ce.jpg      15
1  00091bffdffd83de.jpg      15
2  00396fbc1f6cc21d.jpg      15
3  00471f8038d9c4b6.jpg      15
4  00901f504008d884.jpg      15
5  009b22decbc7220c.jpg      15
6  00b33e0ee6d59427.jpg      15
7  00bbdcfbbdb3e131.jpg      15
8  00c03047e0fbef40.jpg      15
9  00c0dabb63ca7a16.jpg      15
\n======================================================================
✓ 제출 준비 완료!


In [15]:
submission = pd.read_csv('submission_for_test1.csv')

print("=== 클래스 분포 분석 ===")
print(submission['target'].value_counts().sort_index())
print(f"\n가장 많이 예측된 클래스: {submission['target'].mode()[0]}")
print(f"예측 클래스 수: {submission['target'].nunique()}")

# 학습 데이터와 비교
train_df = pd.read_csv('train.csv')
print("\n=== 학습 데이터 분포 ===")
print(train_df['target'].value_counts().sort_index())

=== 클래스 분포 분석 ===
target
15    3140
Name: count, dtype: int64

가장 많이 예측된 클래스: 15
예측 클래스 수: 1

=== 학습 데이터 분포 ===
target
0     100
1      46
2     100
3     100
4     100
5     100
6     100
7     100
8     100
9     100
10    100
11    100
12    100
13     74
14     50
15    100
16    100
Name: count, dtype: int64


In [19]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("테스트: 간단한 CNN 모델 (전이학습 없음)")
print("="*70)

# 데이터 로드
train_df = pd.read_csv('train.csv')
sample_submission = pd.read_csv('sample_submission.csv')

print(f"\n학습 데이터: {len(train_df)}개")

# 이미지 로드 함수
def load_img_simple(img_path):
    try:
        img = load_img(img_path, target_size=(128, 128))
        return np.array(img) / 255.0
    except:
        return np.zeros((128, 128, 3))

# 학습 데이터 로드
print("\n이미지 로드 중...")
train_df['file_path'] = train_df['ID'].apply(lambda x: os.path.join('train', x))
X_train = np.array([load_img_simple(p) for p in train_df['file_path']])
y_train = train_df['target'].values

print(f"✓ X_train shape: {X_train.shape}")

# 간단한 CNN 모델 (전이학습 없음)
print("\n모델 구축 중...")
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(17, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 학습
print("\n모델 학습 중 (5 에포크)...")
history = model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)

print(f"\n최종 정확도: {history.history['accuracy'][-1]:.4f}")

# 테스트 예측
print("\n테스트 예측 중...")
test_files = sorted([f for f in os.listdir('test') if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
test_paths = [os.path.join('test', f) for f in test_files]

test_images = np.array([load_img_simple(p) for p in test_paths])
predictions = model.predict(test_images, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)

# 제출
sample_submission['target'] = predicted_classes
sample_submission.to_csv('submission_simple_cnn.csv', index=False)

print(f"\n✓ 완료!")
print(f"클래스 분포:")
print(pd.Series(predicted_classes).value_counts().sort_index())
print(f"\n첫 20개:")
print(sample_submission.head(20))


테스트: 간단한 CNN 모델 (전이학습 없음)

학습 데이터: 1570개

이미지 로드 중...
✓ X_train shape: (1570, 128, 128, 3)

모델 구축 중...

모델 학습 중 (5 에포크)...
Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 208ms/step - accuracy: 0.3669 - loss: 2.2275
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 208ms/step - accuracy: 0.6968 - loss: 1.0418
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 206ms/step - accuracy: 0.7834 - loss: 0.6904
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 206ms/step - accuracy: 0.8459 - loss: 0.4901
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 205ms/step - accuracy: 0.8554 - loss: 0.4211

최종 정확도: 0.8554

테스트 예측 중...

✓ 완료!
클래스 분포:
0     260
1      49
2     624
3     110
4      54
5     116
6     227
7      66
8     127
9     141
10    504
11    332
12    121
13    166
14     42
15     39
16    162
Name: count, dtype: int64

첫 20개:
                      ID  target
0   0008fdb22ddce0ce.jpg       2
1   00091bffdffd83de.jpg      13
2   00396fbc1f6cc21d.jpg       2
3   00471f8038d9c4b6.jpg       6
4   00901f504008d884.jpg       2
5